### Tramos las 2 tablas bronze: compras y detalles

In [0]:
dbutils.widgets.text("catalog_name", "finalproject")

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')

In [0]:
df_compras = spark.table(f'{catalog_name}.bronze.compras')

df_detalles = spark.table(f'{catalog_name}.bronze.detalles')

## Implementando reglas de calidad

In [0]:
reglas = []

### Reglas de calidad en df_compras

In [0]:
# 1. No null en factura
nulos_factura = df_compras.filter(col('factura').isNull()).count()
reglas.append({
    'tabla': 'df_compras',
    'columna': 'factura',
    'regla': 'no_null',
    'cumple': nulos_factura == 0,
    'detalle': f"Registros nulos: {nulos_factura}"
})

# 2. Facturas con al menos 7 dígitos
menores_a_7_factura = df_compras.filter(length(col('factura')) < 7).count()
reglas.append({
    'tabla': 'df_compras',
    'columna': 'factura',
    'regla': 'Al menos 7 digitos',
    'cumple': menores_a_7_factura == 0,
    'detalle': f"Registros de longitud menor a 7: {menores_a_7_factura}"
})

# 3. Factura no debe contener duplicados
duplicados_factura = df_compras.groupBy('factura').count().filter(col('count') > 1).count()
reglas.append({
    'tabla': 'df_compras',
    'columna': 'factura',
    'regla': 'No duplicados',
    'cumple': duplicados_factura == 0,
    'detalle': f"Registros duplicados: {duplicados_factura}"
})

# 4. No null en fecha_orden
nulos_fecha_orden = df_compras.filter(col('fecha_orden').isNull()).count()
reglas.append({
    'tabla': 'df_compras',
    'columna': 'fecha_orden',
    'regla': 'no_null',
    'cumple': nulos_fecha_orden == 0,
    'detalle': f"Fecha orden nulos: {nulos_fecha_orden}"
})

### Reglas de calidad en df_detalles

In [0]:
# 1. No null en factura
detalles_nulos_factura = df_detalles.filter(col('factura').isNull()).count()
reglas.append({
    'tabla': 'df_detalles',
    'columna': 'factura',
    'regla': 'no_null',
    'cumple': detalles_nulos_factura == 0,
    'detalle': f"Registros nulos: {detalles_nulos_factura}"
})

# 2. No null en producto
detalles_nulos_producto = df_detalles.filter(col('producto').isNull()).count()
reglas.append({
    'tabla': 'df_detalles',
    'columna': 'producto',
    'regla': 'no_null',
    'cumple': detalles_nulos_producto == 0,
    'detalle': f"Registros nulos: {detalles_nulos_producto}"
})

## Ingesta de datos de las reglas

### Convertimos la lista de reglas en dataframe

In [0]:
df_reglas = spark.createDataFrame(reglas)
df_reglas = df_reglas.withColumn('fecha_validacion', current_timestamp())

### Creamos la tabla donde insertaremos los `datos`

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {catalog_name}.log_calidad_datos
(
  id BIGINT GENERATED ALWAYS AS IDENTITY,
  tabla STRING,
  columna STRING,
  regla STRING,
  cumple BOOLEAN,
  detalle STRING,
  fecha_validacion TIMESTAMP
)
          """)

### Insertamos datos en la tabla log_calidad_datos

In [0]:
(
    df_reglas.write\
        .mode('append')\
        .format('delta')\
        .saveAsTable(f'{catalog_name}.bronze.log_calidad_datos')
)

In [0]:
fallas_criticas = [regla for regla in reglas if not regla['cumple']]

if fallas_criticas:
    dbutils.jobs.taskValues.set(key = 'estado', values = 'Falla Critica')
    dbutils.jobs.taskValues.set(key = 'detalle', values = json.dumps(fallas_criticas))
else:
    dbutils.jobs.taskValues.set(key = 'estado', value = 'Validación Exitosa')